# 04 — Feature Availability and Data Leakage Audit

**IT3051 – Fundamentals of Data Mining · Mini Project 2026**

**Prediction point:** The system aims to estimate cancellation risk from reservation information before the final reservation outcome is known. Features that directly reveal the final status or clearly summarize events occurring up to cancellation/check-in must be excluded.

Target leakage means an input directly or indirectly reveals the current reservation's final outcome. Temporal leakage means information includes events after the intended risk assessment. Either can inflate offline performance using evidence unavailable for a real prediction. Timing uncertainty alone is not proof of leakage.

The refined investigation is **Hotel-Type-Specific Cancellation Risk Prediction for City Hotel and Resort Hotel**. Booking-time prediction is not the novelty. Later modelling will investigate whether a general model and hotel-specific models behave differently; superiority is not assumed.

This remains an audit/documentation stage. No preprocessing, feature removal, splitting, or modelling is performed.

## 2 — Load raw data

Use the same repository-root check as earlier notebooks. Record the raw CSV hash and shape. Descriptive evidence is computed from `df`; documentation tables and temporary date Series are separate objects.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
project_root = next((candidate for candidate in (cwd, cwd.parent)
    if (candidate / "requirements.txt").is_file()
    and (candidate / "notebooks").is_dir()
    and (candidate / "data" / "raw").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Run from the repository root or notebooks directory.")
data_path = project_root / "data" / "raw" / "hotel_bookings.csv"
if not data_path.is_file():
    raise FileNotFoundError("Place hotel_bookings.csv inside data/raw/.")
raw_hash_before = hashlib.sha256(data_path.read_bytes()).hexdigest()
df = pd.read_csv(data_path)
original_columns = df.columns.tolist()
original_shape = df.shape
print(f"Dataset: {data_path.relative_to(project_root).as_posix()}; shape: {df.shape}")
print("Raw CSV SHA-256 before audit:", raw_hash_before)
if not {"hotel", "is_canceled"}.issubset(df.columns):
    raise ValueError("Required hotel or target column missing.")
if not df["is_canceled"].isin([0, 1]).all():
    raise ValueError("Review missing or unexpected target labels before proceeding.")

Dataset: data/raw/hotel_bookings.csv; shape: (119390, 32)
Raw CSV SHA-256 before audit: 7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06


## 3 — Feature inventory

List every actual column and its observed dtype. The complete classification table is built in source order after reviewing evidence. No expected feature list is silently substituted for the CSV inventory.

In [2]:
inventory = pd.DataFrame({"Feature": df.columns, "Observed dtype": [str(dtype) for dtype in df.dtypes]})
display(inventory)
print("Actual feature count (including target):", len(inventory))

,Feature,Observed dtype
0,hotel,str
1,is_canceled,int64
2,lead_time,int64
3,arrival_date_year,int64
4,arrival_date_month,str
5,arrival_date_week_number,int64
6,arrival_date_day_of_month,int64
7,stays_in_weekend_nights,int64
8,stays_in_week_nights,int64
9,adults,int64


Actual feature count (including target): 32


## 4 — Classification framework and evidence limits

- **Project role:** Target; Candidate predictor; Context / grouping feature; Identifier-like feature; Outcome-related feature.
- **Leakage risk:** Direct target leakage; Strong temporal leakage risk; Timing / source limitation; No obvious target leakage identified; N/A — target.
- **Proposed modelling status:** Target; Candidate for modelling; Exclude due to direct leakage; Exclude due to strong temporal concern; Review before final preprocessing.

Availability is described relative to a pre-outcome risk assessment, not exact reservation creation. Final outcome information and strong temporal concerns are excluded; timing/source limitations are held for review. Identifier-related quality and feature-selection issues are separate from leakage. Candidate status does not guarantee usefulness or final retention.

The source dataset is retrospective and some booking attributes may reflect modifications after the original reservation was created. This project prevents clear outcome leakage but cannot guarantee that every candidate predictor represents its exact value at booking creation. This retrospective snapshot limitation must be acknowledged in later reporting; an updated attribute is not automatically leakage.

Source definitions: [Antonio, Almeida & Nunes, Hotel booking demand datasets, Sections 1–2 and Table 1](https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/) ([DOI](https://doi.org/10.1016/j.dib.2018.11.126)), together with the project interpretations and local descriptive evidence. No network access is needed to execute this notebook.

## 5 — Direct target and outcome-event leakage

Inspect final-status labels against the target with transparent counts, without a model. Inspect the associated final-status date's parsing and range. Both fields describe the final outcome and must be excluded before modelling. The date alone need not determine the class for its outcome-event meaning to justify exclusion.

In [3]:
status_counts = pd.crosstab(df["reservation_status"], df["is_canceled"], dropna=False)
display(status_counts.rename(columns={0: "Not Cancelled", 1: "Cancelled"}))
status_target_diversity = df.groupby("reservation_status", dropna=False)["is_canceled"].nunique()
print("Distinct target classes per status:")
display(status_target_diversity.to_frame("Target classes"))
status_relationship_confirmed = bool(df["reservation_status"].notna().all() and status_target_diversity.eq(1).all())
print("Every observed status maps to one target class:", status_relationship_confirmed)

status_dates = pd.to_datetime(df["reservation_status_date"], format="%Y-%m-%d", errors="coerce")
date_parse_failures = int((df["reservation_status_date"].notna() & status_dates.isna()).sum())
date_summary = pd.DataFrame([{"Earliest": status_dates.min(), "Latest": status_dates.max(),
    "Distinct dates": status_dates.nunique(), "Missing raw values": df["reservation_status_date"].isna().sum(),
    "Non-null parse failures": date_parse_failures}])
display(date_summary)
display(status_dates.groupby(df["reservation_status"]).agg(["count", "min", "max"]))
display(df[["reservation_status", "reservation_status_date", "is_canceled"]].head(5))
date_class_counts = df.groupby("reservation_status_date", dropna=False)["is_canceled"].nunique()
mixed_target_dates = int(date_class_counts.gt(1).sum())
print("Date values occurring in both target classes:", mixed_target_dates)
display(Markdown("**Proposed exclusions:** `reservation_status` reveals the outcome in the observed cross-tabulation. "
    "`reservation_status_date` records an outcome-related event date, so it is also excluded under the project's direct/outcome-leakage category. "
    "The date's exclusion does not depend on its standalone association strength. Neither column is removed from df."))

is_canceled,Not Cancelled,Cancelled
reservation_status,,
Canceled,0,43017
Check-Out,75166,0
No-Show,0,1207


Distinct target classes per status:


,Target classes
reservation_status,
Canceled,1
Check-Out,1
No-Show,1


Every observed status maps to one target class: True


,Earliest,Latest,Distinct dates,Missing raw values,Non-null parse failures
0,2014-10-17,2017-09-14,926,0,0


,count,min,max
reservation_status,,,
Canceled,43017,2014-10-17,2017-08-26
Check-Out,75166,2015-07-01,2017-09-14
No-Show,1207,2015-07-02,2017-08-31


,reservation_status,reservation_status_date,is_canceled
0,Check-Out,2015-07-01,0
1,Check-Out,2015-07-01,0
2,Check-Out,2015-07-02,0
3,Check-Out,2015-07-02,0
4,Check-Out,2015-07-03,0


Date values occurring in both target classes: 781


**Proposed exclusions:** `reservation_status` reveals the outcome in the observed cross-tabulation. `reservation_status_date` records an outcome-related event date, so it is also excluded under the project's direct/outcome-leakage category. The date's exclusion does not depend on its standalone association strength. Neither column is removed from df.

## 6 — Strong temporal exclusions and timing-review fields

Exclude `booking_changes` because its recorded count accumulates amendments through check-in/cancellation. Exclude `assigned_room_type` because ultimate allocation can reflect later operations or customer changes; it is not the reserved room request. `reserved_room_type` remains a candidate.

Keep three separate timing/source review questions:

- `days_in_waiting_list` can be known after confirmation; suitability depends on whether scoring occurs before or after that confirmation.
- `deposit_type` describes payments identified before arrival/cancellation. It may contain pre-outcome information, but verify which payments were known at assessment.
- `adr` is transaction-derived Average Daily Rate. Verify the source and value available at the intended assessment.

These three fields are not direct target leakage and are not automatically excluded. Descriptive associations alone cannot resolve their timing.

In [4]:
timing_descriptives = df[["booking_changes", "days_in_waiting_list", "adr"]].describe().T
display(timing_descriptives)
room_comparable = df["assigned_room_type"].notna() & df["reserved_room_type"].notna()
room_mismatch = room_comparable & df["assigned_room_type"].ne(df["reserved_room_type"])
timing_counts = pd.DataFrame([
    {"Observation": "Assigned/reserved room codes differ", "Rows": int(room_mismatch.sum())},
    {"Observation": "Room comparison not evaluable", "Rows": int((~room_comparable).sum())},
    {"Observation": "Positive booking changes", "Rows": int(df["booking_changes"].gt(0).sum())},
    {"Observation": "Positive days on waiting list", "Rows": int(df["days_in_waiting_list"].gt(0).sum())},
])
display(timing_counts)
display(df["assigned_room_type"].value_counts(dropna=False).to_frame("Bookings"))
display(df["deposit_type"].value_counts(dropna=False).to_frame("Bookings"))
display(Markdown("Matching room codes or zero booking changes do not remove the strong field-level timing concern. "
    "Waiting duration, deposit type, and ADR remain review-required rather than automatically excluded."))

,count,mean,std,min,25%,50%,75%,max
booking_changes,119390.0,0.221124,0.652306,0.00,0.00,0.000,0.0,21.0
days_in_waiting_list,119390.0,2.321149,17.594721,0.00,0.00,0.000,0.0,391.0
adr,119390.0,101.831122,50.535790,-6.38,69.29,94.575,126.0,5400.0


,Observation,Rows
0,Assigned/reserved room codes differ,14917
1,Room comparison not evaluable,0
2,Positive booking changes,18076
3,Positive days on waiting list,3698


,Bookings
assigned_room_type,
A,74053
D,25322
E,7806
F,3751
G,2553
C,2375
B,2163
H,712
I,363


,Bookings
deposit_type,
No Deposit,104641
Non Refund,14587
Refundable,162


Matching room codes or zero booking changes do not remove the strong field-level timing concern. Waiting duration, deposit type, and ADR remain review-required rather than automatically excluded.

## 7 — Identifier-like features

`agent` and `company` are identifier-like candidate features requiring preprocessing/feature-selection review, not leakage exclusions. Numeric-looking codes are not continuous measurements. They may contain useful information, but high cardinality, missing/not-applicable values, suitable encoding, and possible later exclusion require review. Missing values are not assumed to be errors or automatically assigned a meaning. No encoding or selection is performed.

In [5]:
identifier_summary = pd.DataFrame([
    {"Feature": column, "Observed dtype": str(df[column].dtype),
     "Unique non-null codes": df[column].nunique(), "Missing count": int(df[column].isna().sum()),
     "Missing percentage": 100 * df[column].isna().mean()}
    for column in ["agent", "company"]
])
display(identifier_summary.round(3))

,Feature,Observed dtype,Unique non-null codes,Missing count,Missing percentage
0,agent,float64,333,16340,13.686
1,company,float64,352,112593,94.307


## 8 — Previous customer history

`is_repeated_guest`, `previous_cancellations`, and `previous_bookings_not_canceled` refer to customer information/history prior to the current booking. Under those definitions they are candidates, not future information for the current cancellation target. This classification does not require retaining them after quality, redundancy, or usefulness review.

In [6]:
history_features = ["previous_cancellations", "previous_bookings_not_canceled", "is_repeated_guest"]
display(df[history_features].describe().T)
display(df["is_repeated_guest"].value_counts(dropna=False).to_frame("Bookings"))

,count,mean,std,min,25%,50%,75%,max
previous_cancellations,119390.0,0.087118,0.844336,0.0,0.0,0.0,0.0,26.0
previous_bookings_not_canceled,119390.0,0.137097,1.497437,0.0,0.0,0.0,0.0,72.0
is_repeated_guest,119390.0,0.031912,0.175767,0.0,0.0,0.0,0.0,1.0


,Bookings
is_repeated_guest,
0,115580
1,3810


## 9 — Reservation-detail availability policy

Ordinary reservation attributes remain candidate predictors with the retrospective snapshot limitation documented globally. A value may have been amended without revealing the final target. Availability uncertainty is not equivalent to target leakage.

Availability and usefulness are separate. Quality, redundancy, high cardinality, or limited predictive value may justify later selection decisions, but those decisions are not made in this audit. Proposed status controls the feature lists; a conceptual candidate role alone does not approve a review-required field.

In [7]:
policy = {}
def classify(feature, role, availability, leakage, status, reason):
    policy[feature] = {"Feature": feature, "Project role": role, "Availability category": availability,
        "Leakage category": leakage, "Proposed modelling status": status, "Evidence / reasoning": reason}

classify("is_canceled", "Target", "N/A — target", "N/A — target", "Target", "Current reservation outcome; never an input predictor.")
normal_reasons = {
    "hotel": "Hotel type supplies context and the City/Resort grouping; it is constant within a hotel-specific subset.",
    "lead_time": "Reservation timing attribute; amendments to arrival plans are a snapshot limitation, not automatic outcome leakage.",
    "arrival_date_year": "Arrival-plan year; a potentially updated plan does not directly reveal the target.",
    "arrival_date_month": "Arrival-plan month; subject to the documented retrospective snapshot limitation.",
    "arrival_date_week_number": "Arrival-plan week; subject to the documented retrospective snapshot limitation.",
    "arrival_date_day_of_month": "Arrival-plan day; subject to the documented retrospective snapshot limitation.",
    "stays_in_weekend_nights": "Reservation stay-duration attribute; updates alone do not establish target leakage.",
    "stays_in_week_nights": "Reservation stay-duration attribute; updates alone do not establish target leakage.",
    "adults": "Guest-count attribute; amendments and unusual counts belong to snapshot/quality limitations.",
    "children": "Guest-count attribute; missingness and amendments do not by themselves establish leakage.",
    "babies": "Guest-count attribute; no obvious target leakage identified under the revised scope.",
    "meal": "Reservation meal category; updates alone are not grounds for a leakage exclusion.",
    "country": "Customer/reservation information; corrections, missingness, and cardinality need ordinary quality review.",
    "market_segment": "Reservation market classification; no direct final-outcome information identified.",
    "distribution_channel": "Booking-channel classification; no obvious target leakage identified.",
    "reserved_room_type": "Reserved room request remains a candidate and is distinct from the ultimate room assignment.",
    "customer_type": "Customer/booking category; possible updates are a documented snapshot limitation.",
    "required_car_parking_spaces": "Reservation request count; updates alone do not prove leakage.",
    "total_of_special_requests": "Reservation request count; possible updates are a snapshot limitation, not a proven outcome summary.",
}
for feature, reason in normal_reasons.items():
    classify(feature, "Context / grouping feature" if feature == "hotel" else "Candidate predictor",
        "Reservation information; retrospective snapshot limitation", "No obvious target leakage identified", "Candidate for modelling", reason)
for feature in history_features:
    classify(feature, "Candidate predictor", "Prior customer history", "No obvious target leakage identified", "Candidate for modelling",
        "Definition refers to customer information/history before the current booking, not its future cancellation outcome. Final retention remains a later selection decision.")

status_evidence = "; ".join(f"{status}: " + ", ".join(f"target {int(label)}={int(count):,}" for label, count in row.items())
    for status, row in status_counts.iterrows())
classify("reservation_status", "Outcome-related feature", "Final-outcome information", "Direct target leakage", "Exclude due to direct leakage",
    "Final reservation status describes the outcome. Observed cross-tabulation: " + status_evidence + ".")
classify("reservation_status_date", "Outcome-related feature", "Final-outcome information", "Direct target leakage", "Exclude due to direct leakage",
    "Date associated with final status is outcome-related information; exclusion does not imply the date alone determines the target.")
classify("assigned_room_type", "Candidate predictor", "Later operational assignment", "Strong temporal leakage risk", "Exclude due to strong temporal concern",
    f"Ultimately assigned room may reflect later operations/customer changes; {int(room_mismatch.sum()):,} codes differ from reserved codes. Not equivalent to reserved_room_type.")
classify("booking_changes", "Candidate predictor", "Events accumulated through check-in/cancellation", "Strong temporal leakage risk", "Exclude due to strong temporal concern",
    f"Final count accumulates amendments from entry until check-in/cancellation and may summarize future events; {int(df['booking_changes'].gt(0).sum()):,} positive counts.")
review_reasons = {
    "days_in_waiting_list": "Elapsed time between entry and confirmation can be known after confirmation; suitability depends on the operational assessment point. Not direct leakage or an automatic exclusion.",
    "deposit_type": "Based on payments identified before arrival/cancellation; may supply useful pre-outcome information. Review payment timing relative to assessment; do not automatically exclude.",
    "adr": "Average Daily Rate is derived from lodging transactions. Verify the source and value known at assessment; retrospective uncertainty is not confirmed temporal leakage or automatic exclusion.",
}
for feature, reason in review_reasons.items():
    classify(feature, "Candidate predictor", "Assessment-time availability needs review", "Timing / source limitation", "Review before final preprocessing", reason)
for feature in ["agent", "company"]:
    info = identifier_summary.set_index("Feature").loc[feature]
    classify(feature, "Identifier-like feature", "Reservation-associated identifier", "No obvious target leakage identified", "Review before final preprocessing",
        f"Identifier-like candidate: {int(info['Unique non-null codes']):,} observed codes; {int(info['Missing count']):,} missing ({info['Missing percentage']:.3f}%). Review missing/not-applicable meaning, encoding and usefulness or possible exclusion later; this is not a leakage decision.")

if set(policy) != set(df.columns):
    raise ValueError("Dataset schema changed: revise the feature-by-feature policy before proceeding.")
audit = pd.DataFrame([policy[feature] for feature in df.columns])
audit.insert(1, "Observed dtype", [str(df[feature].dtype) for feature in df.columns])
display(audit.loc[audit["Feature"].isin(normal_reasons), ["Feature", "Availability category", "Proposed modelling status", "Evidence / reasoning"]])

,Feature,Availability category,Proposed modelling status,Evidence / reasoning
0,hotel,Reservation information; retrospective snapsho...,Candidate for modelling,Hotel type supplies context and the City/Resor...
2,lead_time,Reservation information; retrospective snapsho...,Candidate for modelling,Reservation timing attribute; amendments to ar...
3,arrival_date_year,Reservation information; retrospective snapsho...,Candidate for modelling,Arrival-plan year; a potentially updated plan ...
4,arrival_date_month,Reservation information; retrospective snapsho...,Candidate for modelling,Arrival-plan month; subject to the documented ...
5,arrival_date_week_number,Reservation information; retrospective snapsho...,Candidate for modelling,Arrival-plan week; subject to the documented r...
6,arrival_date_day_of_month,Reservation information; retrospective snapsho...,Candidate for modelling,Arrival-plan day; subject to the documented re...
7,stays_in_weekend_nights,Reservation information; retrospective snapsho...,Candidate for modelling,Reservation stay-duration attribute; updates a...
8,stays_in_week_nights,Reservation information; retrospective snapsho...,Candidate for modelling,Reservation stay-duration attribute; updates a...
9,adults,Reservation information; retrospective snapsho...,Candidate for modelling,Guest-count attribute; amendments and unusual ...
10,children,Reservation information; retrospective snapsho...,Candidate for modelling,Guest-count attribute; missingness and amendme...


## 10 — Target check

Verify the target has the exact target labels in all policy fields and cannot enter a predictor list.

In [8]:
target_row = audit.set_index("Feature").loc["is_canceled"]
assert target_row["Project role"] == "Target"
assert target_row["Availability category"] == "N/A — target"
assert target_row["Leakage category"] == "N/A — target"
assert target_row["Proposed modelling status"] == "Target"
display(target_row.to_frame("Target policy"))

,Target policy
Observed dtype,int64
Project role,Target
Availability category,N/A — target
Leakage category,N/A — target
Proposed modelling status,Target
Evidence / reasoning,Current reservation outcome; never an input pr...


## 11 — Complete leakage evidence table

Every actual dataset column appears once. Roles describe conceptual purpose; status controls the proposed modelling policy. “Requires verification” is not silently treated as safe.

In [9]:
evidence_table = audit[["Feature", "Availability category", "Leakage category", "Proposed modelling status", "Evidence / reasoning"]].rename(
    columns={"Availability category": "Availability", "Leakage category": "Leakage Risk", "Proposed modelling status": "Proposed Status", "Evidence / reasoning": "Reason"})
with pd.option_context("display.max_rows", None, "display.max_colwidth", 110):
    display(evidence_table)
assert audit["Feature"].tolist() == df.columns.tolist()
assert audit["Feature"].is_unique
assert len(audit) == 32, "Dataset schema differs from the 32-column project dataset; review the audit."

,Feature,Availability,Leakage Risk,Proposed Status,Reason
0,hotel,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,Hotel type supplies context and the City/Resort grouping; it is constant within a hotel-specific subset.
1,is_canceled,N/A — target,N/A — target,Target,Current reservation outcome; never an input predictor.
2,lead_time,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,"Reservation timing attribute; amendments to arrival plans are a snapshot limitation, not automatic outcome..."
3,arrival_date_year,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,Arrival-plan year; a potentially updated plan does not directly reveal the target.
4,arrival_date_month,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,Arrival-plan month; subject to the documented retrospective snapshot limitation.
5,arrival_date_week_number,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,Arrival-plan week; subject to the documented retrospective snapshot limitation.
6,arrival_date_day_of_month,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,Arrival-plan day; subject to the documented retrospective snapshot limitation.
7,stays_in_weekend_nights,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,Reservation stay-duration attribute; updates alone do not establish target leakage.
8,stays_in_week_nights,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,Reservation stay-duration attribute; updates alone do not establish target leakage.
9,adults,Reservation information; retrospective snapshot limitation,No obvious target leakage identified,Candidate for modelling,Guest-count attribute; amendments and unusual counts belong to snapshot/quality limitations.


## 12 — Proposed feature sets

These lists document the policy without transforming `df`. They form a mutually exclusive, exhaustive partition. `agent` and `company` are conceptual identifier-like candidates, listed in the review set because treatment is not decided. The other review fields have specific timing/source questions, not automatic exclusions.

In [10]:
def features_with_status(status):
    return audit.loc[audit["Proposed modelling status"].eq(status), "Feature"].tolist()

target_feature = features_with_status("Target")
candidate_predictors = features_with_status("Candidate for modelling")
direct_leakage_features = features_with_status("Exclude due to direct leakage")
timing_excluded_features = features_with_status("Exclude due to strong temporal concern")
review_required_features = features_with_status("Review before final preprocessing")
feature_sets = {
    "target_feature": target_feature,
    "candidate_predictors": candidate_predictors,
    "direct_leakage_features": direct_leakage_features,
    "timing_excluded_features": timing_excluded_features,
    "review_required_features": review_required_features,
}
for name, features in feature_sets.items():
    print(f"{name} ({len(features)}): {features}")
flat_features = [feature for features in feature_sets.values() for feature in features]
assert len(flat_features) == len(set(flat_features)) == len(df.columns)
assert set(flat_features) == set(df.columns)
assert target_feature == ["is_canceled"]
assert not set(candidate_predictors) & set(direct_leakage_features)
assert not set(candidate_predictors) & set(target_feature)
assert not set(candidate_predictors) & set(timing_excluded_features + review_required_features)
assert set(direct_leakage_features) == {"reservation_status", "reservation_status_date"}
assert set(timing_excluded_features) == {"assigned_room_type", "booking_changes"}
assert set(review_required_features) == {"adr", "deposit_type", "days_in_waiting_list", "agent", "company"}
assert set(normal_reasons).issubset(candidate_predictors)
assert set(history_features).issubset(candidate_predictors)
assert audit.set_index("Feature").loc[["adr", "deposit_type", "days_in_waiting_list"], "Leakage category"].eq("Timing / source limitation").all()
assert audit.set_index("Feature").loc[["agent", "company"], "Leakage category"].eq("No obvious target leakage identified").all()

target_feature (1): ['is_canceled']
candidate_predictors (22): ['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'customer_type', 'required_car_parking_spaces', 'total_of_special_requests']
direct_leakage_features (2): ['reservation_status', 'reservation_status_date']
timing_excluded_features (2): ['assigned_room_type', 'booking_changes']
review_required_features (5): ['deposit_type', 'agent', 'company', 'days_in_waiting_list', 'adr']


## 13 — Why this matters for project validity

A model using final status could read the answer, while accumulated final-event information could make offline performance unrealistic for an earlier assessment. Preventing those clear leakage paths is more meaningful than maximising apparent accuracy.

Both the general and hotel-specific models must later use the same leakage-controlled feature policy. This supports fair comparison without assuming that separate models will be better. Normal reservation attributes remain candidates; their possible amendment is acknowledged rather than equated with leakage.

The source dataset is retrospective and some booking attributes may reflect modifications after the original reservation was created. This project prevents clear outcome leakage but cannot guarantee that every candidate predictor represents its exact value at booking creation. This retrospective snapshot limitation must be acknowledged in later reporting; an updated attribute is not automatically leakage.

The refined investigation is **Hotel-Type-Specific Cancellation Risk Prediction for City Hotel and Resort Hotel**. Booking-time prediction is not the novelty. Later modelling will investigate whether a general model and hotel-specific models behave differently; superiority is not assumed.

## 14 — Final audit summary and integrity

Summarise the proposed policy, retain unresolved uncertainty, and confirm both the file and DataFrame are unchanged. No charts or machine-learning measures are needed for this audit.

In [11]:
def bullets(values):
    return "\n".join(f"- `{value}`" for value in values) or "None."

summary = (f"**Direct/outcome leakage ({len(direct_leakage_features)}):** " + ", ".join(direct_leakage_features)
    + f".\n\n**Timing exclusions ({len(timing_excluded_features)}):** " + ", ".join(timing_excluded_features)
    + f".\n\n**Candidate predictors ({len(candidate_predictors)}):** " + ", ".join(candidate_predictors)
    + f".\n\n**Review required ({len(review_required_features)}):** " + ", ".join(review_required_features)
    + ".\n\nADR, deposit type, and waiting duration require assessment-time/source review. Agent/company need preprocessing/feature-selection review, not leakage exclusion. "
    "Normal attributes and prior history remain candidates with the retrospective snapshot limitation documented. No preprocessing or final feature selection has occurred.")
display(Markdown(summary))

assert df.shape == original_shape and df.columns.tolist() == original_columns
pd.testing.assert_frame_equal(df, pd.read_csv(data_path))
raw_hash_after = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert raw_hash_after == raw_hash_before
print("Raw CSV SHA-256 after audit:", raw_hash_after)
print("Verified: all 32 features classified; no feature-set overlap; raw file and df unchanged.")

**Direct/outcome leakage (2):** reservation_status, reservation_status_date.

**Timing exclusions (2):** assigned_room_type, booking_changes.

**Candidate predictors (22):** hotel, lead_time, arrival_date_year, arrival_date_month, arrival_date_week_number, arrival_date_day_of_month, stays_in_weekend_nights, stays_in_week_nights, adults, children, babies, meal, country, market_segment, distribution_channel, is_repeated_guest, previous_cancellations, previous_bookings_not_canceled, reserved_room_type, customer_type, required_car_parking_spaces, total_of_special_requests.

**Review required (5):** deposit_type, agent, company, days_in_waiting_list, adr.

ADR, deposit type, and waiting duration require assessment-time/source review. Agent/company need preprocessing/feature-selection review, not leakage exclusion. Normal attributes and prior history remain candidates with the retrospective snapshot limitation documented. No preprocessing or final feature selection has occurred.

Raw CSV SHA-256 after audit: 7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06
Verified: all 32 features classified; no feature-set overlap; raw file and df unchanged.
